哈特曼传感器

光路参数初始化  

In [1]:
from pprint import pprint
import numpy as np

# All physical lengths use SI units (m) unless the name explicitly says px/mm/um.
optical_params = {
    "layout": [
        "561 nm laser",
        "aperture",
        "L1/L2 4f relay",
        "flat mirror now / DM later",
        "L3/L4 4f relay",
        "Shack-Hartmann sensor",
    ],
    "source": {
        "laser_wavelength_m": 561.0e-9,
        "description": "561 nm laser -> aperture -> L1/L2 4f -> mirror/DM -> L3/L4 4f -> Shack-Hartmann sensor",
    },
    "camera": {
        "resolution_px": (1440, 1080),
        "pixel_pitch_m": 3.45e-6,
    },
    "dm": {
        "enabled": True,
        "Nact": 12,
        "DMmesh": 312,
        "DMstride": 26,
        "DMgain_m_per_count": 15.372e-9,
        "wavelength_m": 680.0e-9,
        "mirror_factor": 2.0,
        "pitch_m": 400.0e-6,
        "sigma": 1.125,
        "width": 5,
    },
    "hartmann": {
        # Fill these after checking the exact MLA datasheet / optical layout.
        "lenslet_pitch_m": None,
        "lenslet_focal_length_m": None,
        "l3_l4_magnification": None,
        "target_spot_field_coverage": (0.70, 0.90),
    },
    "spot_tracking": {
        "max_expected_shift_px": 20,
        "spot_radius_px": 5,
        "safety_margin_px": 10,
        "search_window_radius_px": 30,
    },
}

# Derived camera quantities.
camera_w_px, camera_h_px = optical_params["camera"]["resolution_px"]
pixel_pitch_m = optical_params["camera"]["pixel_pitch_m"]
sensor_size_m = np.array([camera_w_px, camera_h_px], dtype=float) * pixel_pitch_m
sensor_diag_m = float(np.linalg.norm(sensor_size_m))

# Derived DM aperture estimates from notes.md.
dm = optical_params["dm"]
actuator_center_span_m = (dm["Nact"] - 1) * dm["pitch_m"]
dm_program_aperture_m = dm["Nact"] * dm["pitch_m"]
dm_grid_check = dm["DMmesh"] / dm["DMstride"]

# Recommended edge clearance for valid reference spots.
tracking = optical_params["spot_tracking"]
min_edge_clearance_px = (
    tracking["max_expected_shift_px"]
    + tracking["spot_radius_px"]
    + tracking["safety_margin_px"]
)
safe_sensor_roi_px = (
    min_edge_clearance_px,
    min_edge_clearance_px,
    camera_w_px - min_edge_clearance_px,
    camera_h_px - min_edge_clearance_px,
)

derived_params = {
    "sensor_size_mm": tuple(float(v) for v in sensor_size_m * 1e3),
    "sensor_diagonal_mm": sensor_diag_m * 1e3,
    "dm_actuator_center_span_mm": actuator_center_span_m * 1e3,
    "dm_program_aperture_mm": dm_program_aperture_m * 1e3,
    "dm_grid_actuator_count_check": dm_grid_check,
    "min_edge_clearance_px": min_edge_clearance_px,
    "safe_sensor_roi_px": safe_sensor_roi_px,
}

pprint(optical_params)
print("\nDerived parameters:")
pprint(derived_params)


{'camera': {'pixel_pitch_m': 3.45e-06, 'resolution_px': (1440, 1080)},
 'dm': {'DMgain_m_per_count': 1.5372e-08,
        'DMmesh': 312,
        'DMstride': 26,
        'Nact': 12,
        'enabled': True,
        'mirror_factor': 2.0,
        'pitch_m': 0.0004,
        'sigma': 1.125,
        'wavelength_m': 6.8e-07,
        'width': 5},
 'hartmann': {'l3_l4_magnification': None,
              'lenslet_focal_length_m': None,
              'lenslet_pitch_m': None,
              'target_spot_field_coverage': (0.7, 0.9)},
 'layout': ['561 nm laser',
            'aperture',
            'L1/L2 4f relay',
            'flat mirror now / DM later',
            'L3/L4 4f relay',
            'Shack-Hartmann sensor'],
 'source': {'description': '561 nm laser -> aperture -> L1/L2 4f -> mirror/DM '
                           '-> L3/L4 4f -> Shack-Hartmann sensor',
            'laser_wavelength_m': 5.61e-07},
 'spot_tracking': {'max_expected_shift_px': 20,
                   'safety_margin_px': 10,


下面使用初始化一组zernike系数，并使用DM生成这组zernike系数的图像。